[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/02b_feasibility_preflight.ipynb)

# Can I fine-tune this? — feasibility & pre-flight

> **Fine-tuning series — 02b of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Can I fine-tune this? — feasibility & pre-flight

"Can I fine-tune *this model* with *these settings* for *this dataset*?" splits into three
questions. Only the first is answerable in advance; the other two you **confirm with a quick
test**, not by predicting.

1. **Will it fit in memory?** — calculable (estimator below).
2. **Does the data/method match?** — judgment: right format for the objective, enough data for
   the *kind* of change (behavior vs. new knowledge), and clean/consistent quality.
3. **Is it actually learning?** — proven by a smoke test + validation, not assumed.

### 1. Will it fit? — VRAM estimator

Rough rule of thumb: full FT ≈ **16 bytes/param** (weights + grad + fp32 master + Adam m,v);
LoRA freezes the base (16-bit) so the overhead lands only on the ~1% adapters; QLoRA stores
the base in **4-bit**. Activations vary with batch/sequence length, so this adds a headroom
buffer — treat it as a ballpark, not a guarantee.

In [ ]:
def vram_gb(params_b, method, dtype="bf16", headroom=0.25):
    P = params_b * 1e9
    bpp = {"fp32": 4, "bf16": 2, "fp16": 2}[dtype]
    if method == "full":
        need = P * 16                       # weights + grad + fp32 master + Adam m,v
    elif method == "lora":
        need = P * bpp + 0.01 * P * 16      # frozen 16-bit base + optimizer on ~1% adapters
    elif method == "qlora":
        need = P * 0.5 + 0.01 * P * 16      # 4-bit NF4 base + optimizer on ~1% adapters
    return need / 1e9 * (1 + headroom)      # + activation/headroom buffer

def verdict(gb):
    for cap, name in [(16,"16GB"), (24,"24GB"), (40,"40GB"), (48,"48GB"), (80,"80GB")]:
        if gb <= cap: return f"fits a {name} GPU"
    return "needs multi-GPU / sharding"

print(f"{'model':7s} {'method':7s} {'~VRAM':>9s}   verdict")
for pb in (0.5, 1, 7, 13, 70):            # <- change to your model size (billions of params)
    for m in ("full", "lora", "qlora"):
        gb = vram_gb(pb, m)
        print(f"{str(pb)+'B':7s} {m:7s} {gb:7.1f} GB   {verdict(gb)}")
    print()

### 2. Does the data/method match? (judgment)

- **Format matches the objective** — SFT `{prompt, answer}`, DPO `{prompt, chosen, rejected}`,
  etc. (see the data-formats table in Foundations). Wrong shape → errors or learns nothing.
- **Size vs. goal** — teaching a *behavior/format/tone* works with hundreds–few-thousand
  examples; teaching *substantial new knowledge* needs far more (or continued pretraining, or
  RAG). Too few examples + high rank → overfitting.
- **Quality** — clean, consistent, correctly chat-templated. Garbage in, garbage out.

### 3. Is it actually learning? — the smoke test

Before a long run, deliberately try to **overfit ~8 examples** for a few steps. The loss
should crash toward 0. If it *can't even memorize 8 examples*, the pipeline is broken (wrong
template, LR too low, or a dead adapter) — not the model or dataset. Also confirm
`print_trainable_parameters()` is **> 0**: `0%` means the adapter attached to nothing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

smoke = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
tiny = sft_ds.select(range(min(8, len(sft_ds))))          # just a handful of examples

tr = SFTTrainer(model=smoke, train_dataset=tiny, peft_config=lora_cfg,
    args=SFTConfig(output_dir="smoke", dataset_text_field="text",
                   per_device_train_batch_size=2, max_steps=40, learning_rate=2e-4,
                   logging_steps=5, bf16=BF16_OK, report_to="none"))
tr.model.print_trainable_parameters()                     # MUST be > 0
tr.train()

losses = [h["loss"] for h in tr.state.log_history if "loss" in h]
print(f"\nloss: {losses[0]:.3f}  ->  {losses[-1]:.3f}")
print("Crashed toward ~0? pipeline works. Flat? check template / LR / that trainable > 0.")
del tr, smoke; cleanup()

### Pre-flight checklist

1. Memory fits (estimator above)?
2. Data formatted to the objective + clean?
3. Enough data for the *kind* of change you want (behavior vs. knowledge)?
4. `print_trainable_parameters() > 0`?
5. Smoke test: overfit ~8 examples → loss → ~0?
6. Real run: train loss ↓ **and** validation metric ↑?

Pass 1–6 and the answer is "yes, this works." Fail #5 → fix the pipeline; #6 shows overfitting
→ lower rank / add data. That's the feedback loop, not a dead end.